# 5. Validation: Literature Cases

We compare the implicit solver to published results for typical lubrication problems: first a hydrodynamic case without cavitation, then several cases with cavitation.

Markers in the plots show the reference data, lines the implicit solver solution.

In [ ]:
from GaPFlow.problem import Problem
from utils import load_template, run_pressure_profile
from utils.plotting import plot_almqvist_validation, plot_overview_1d
from utils.topography import (
    regen_twin_parabolic_slider,
    regen_twin_parabolic_slider_id,
    regen_conv_slider_pocket_1d,
    regen_conv_slider_pocket_1d_hansen,
)

REF_DIR = '../../tests/ref_data'

## Inclined Slider (Almqvist)

Linear converging gap with a power-law equation of state, for three sliding velocities. The exponent $\alpha$ controls the compressibility: $\alpha = 0$ is compressible, $\alpha \to 2$ is incompressible-like (small density changes, large pressures).

<img src="./figures/Geometry_InclinedSlider.png"
     alt="Inclined slider geometry"
     width="250">

$\rightarrow$ **[YAML File](../../config_lib/validation/inclined_slider.yaml)**

In [ ]:
template = load_template('inclined_slider')
results = {}
for U in [5., 50., 125.]:
    for kind, alpha in [('comp', 0.), ('incomp', 1.9999)]:
        results[(U, kind)] = run_pressure_profile(template, **{'geometry.U': U, 'properties.alpha': alpha})
plot_almqvist_validation(results)

**Cavitation**

The following cases include cavitation, modelled with the mass-conserving Elrod-Adams formulation (fill fraction $\theta$, Fischer-Burmeister complementarity) and stabilized as specified in the respective YAML file. The plots show pressure and effective density $\rho_\mathrm{eff} = \rho(1-\theta)$.

The geometries of the twin sliders and of the pockets cannot be specified in the YAML file and are set from Python before the run.

## Parabolic Slider (Bayada)

<img src="./figures/Geometry_ParabolicSlider.png"
     alt="Parabolic slider geometry"
     width="250">

$\rightarrow$ **[YAML File](../../config_lib/validation/parabolic_slider_cav.yaml)**

In [ ]:
problem = Problem.from_yaml('../../config_lib/validation/parabolic_slider_cav.yaml')
problem.run()
plot_overview_1d(problem, ref_csv=f'{REF_DIR}/ref_parabolic_slider.csv', ref_label='Bayada',
                 ref_x_scale=problem.grid['Lx'])

## Parabolic Slider (Sahlin)

$\rightarrow$ **[YAML File](../../config_lib/validation/parabolic_slider_cav_sahlin.yaml)**

In [ ]:
problem = Problem.from_yaml('../../config_lib/validation/parabolic_slider_cav_sahlin.yaml')
problem.run()
plot_overview_1d(problem, ref_csv=f'{REF_DIR}/Sahlin_parab_p.csv', ref_label='Sahlin',
                 ref_rho_csv=f'{REF_DIR}/Sahlin_parab_theta.csv',
                 ref_p_scale=1e6, p_unit='MPa', p_divisor=1e6)

## Twin Parabolic Slider, Non-Identical (Bayada)

<img src="./figures/Geometry_TwinParabolicSlider.png"
     alt="Twin parabolic slider geometry"
     width="250">

Note: the simulation-literature mismatch is probably due to wrong reference data. Important point is that no second pressure peak is formed.

This is a case where OSS performs better than UPG. The pressure oscillations at film reformulation are an expected artefact.

$\rightarrow$ **[YAML File](../../config_lib/validation/twin_parabolic_slider.yaml)**

In [ ]:
problem = Problem.from_yaml('../../config_lib/validation/twin_parabolic_slider.yaml')
regen_twin_parabolic_slider(problem)
problem.run()
plot_overview_1d(problem, ref_csv=f'{REF_DIR}/ref_twin_parabolic_slider_ori.csv', ref_label='Bayada',
                 ref_x_scale=0.09, ref_p_scale=1e5)

## Twin Parabolic Slider, Identical (Sahlin)

<img src="./figures/Geometry_TwinParabolicSliderIdentical.png"
     alt="Twin identical parabolic slider geometry"
     width="250">

$\rightarrow$ **[YAML File](../../config_lib/validation/twin_parabolic_slider_id.yaml)**

In [ ]:
problem = Problem.from_yaml('../../config_lib/validation/twin_parabolic_slider_id.yaml')
regen_twin_parabolic_slider_id(problem)
problem.run()
plot_overview_1d(problem, ref_csv=f'{REF_DIR}/ref_twin_parabolic_slider_id.csv', ref_label='Sahlin',
                 ref_x_scale=problem.grid['Lx'], ref_p_scale=1e6, p_unit='MPa', p_divisor=1e6)

## Convergent Slider with Pocket (Bertocchi)

<img src="./figures/Geometry_ConvSliderPocket.png"
     alt="Convergent slider with pocket geometry"
     width="250">

$\rightarrow$ **[YAML File](../../config_lib/validation/conv_slider_pocket_1D.yaml)**

In [ ]:
problem = Problem.from_yaml('../../config_lib/validation/conv_slider_pocket_1D.yaml')
regen_conv_slider_pocket_1d(problem, Nx_geo=50)
problem.run()
plot_overview_1d(problem, ref_csv=f'{REF_DIR}/ref_conv_slider_pocket_1D.csv', ref_label='Bertocchi',
                 ref_x_scale=0.020, ref_p_scale=1e6, p_unit='MPa', p_divisor=1e6)

## Convergent Slider with Pocket (Hansen)

Same type of geometry with different proportions.

$\rightarrow$ **[YAML File](../../config_lib/validation/conv_slider_pocket_1D_hansen.yaml)**

In [ ]:
problem = Problem.from_yaml('../../config_lib/validation/conv_slider_pocket_1D_hansen.yaml')
regen_conv_slider_pocket_1d_hansen(problem, Nx_geo=40)
problem.run()
plot_overview_1d(problem, p_unit='MPa', p_divisor=1e6)